In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [3]:
# DB Station data
db_stations = pd.read_csv(PATH_RAW / 'db_station_data.csv')
db_stations = db_stations[db_stations['Bundesland'] == 'Berlin'].copy()
display(db_stations)

# Fix station names for matching
db_stations[db_stations['Bahnhof'].str.startswith('lin ')] = db_stations[db_stations['Bahnhof'].str.startswith('lin ')].copy().assign(Bahnhof=lambda df: df['Bahnhof'].str[4:])

,Bf-Nr,Aufgabenträger,Bahnhof,klasse,Bundesland,Anteil Serviceeinrichtung Stationspreis SPNV,Anteil Serviceeinrichtung Stationspreis SPFV
29,28,VBB Berlin,Ahrensfelde,4,Berlin,2.06,5.92
41,45,VBB Berlin,Albrechtshof,5,Berlin,2.02,5.77
51,53,VBB Berlin,Alexanderplatz,3,Berlin,3.93,11.32
101,116,VBB Berlin,Altglienicke,4,Berlin,2.06,5.92
108,7719,VBB Berlin,Alt-Reinickendorf,5,Berlin,2.02,5.77
...,...,...,...,...,...,...,...
5266,6824,VBB Berlin,Wittenau (Wilhelmsruher Damm),4,Berlin,2.06,5.92
5295,6871,VBB Berlin,Wollankstraße,4,Berlin,2.06,5.92
5312,6899,VBB Berlin,Wuhlheide,5,Berlin,2.02,5.77
5356,6967,VBB Berlin,Yorckstraße,5,Berlin,2.02,5.77


In [4]:
# The OSM stations query has been moved into osm.py as `_STATIONS_TAGS`.
# It fetches all elements tagged railway=station (S-Bahn, U-Bahn, regional,
# and long-distance rail) using `out geom tags;` — equivalent to the original
# query but with inline geometry, so no manual node-ref assembly is needed.
#
# Results are cached to data/raw/OSM_POIs_Berlin_stations.parquet.

from hotelling.spatial.osm import fetch_pois

osm_stations = fetch_pois(type="stations", city="Berlin")
print(f"OSM stations: {len(osm_stations)} rows, {osm_stations.shape[1]} columns")
print(f"Columns: {list(osm_stations.columns)}")
osm_stations.head()


OSM stations: 283 rows, 116 columns
Columns: ['osm_id', 'osm_type', 'geometry', 'contact:website', 'light_rail', 'name', 'network', 'network:short', 'network:wikidata', 'official_name', 'operator', 'public_transport', 'railway', 'railway:ref', 'railway:station_category', 'station', 'train', 'uic_ref', 'wheelchair', 'wikidata', 'wikipedia', 'contact:phone', 'departures_board', 'departures_board:speech_output', 'description', 'ref', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'ref:ibnr', 'ref:station', 'uic_name', 'website', 'loc_name', 'note', 'start_date', 'toilets:wheelchair', 'subway', 'image', 'source', 'line', 'atm', 'atm:operator', 'operator:short', 'operator:wikidata', 'check_date:wheelchair', 'level', 'internet_access', 'surveillance', 'internet_access:fee', 'railway:ref:parent', 'source:address', 'phone', 'wikimedia_commons', 'surveillance:type', 'name:VBB', 'wheelchair:description', 'wheelchair:source', 'old_name', 'not:network:wikidata', 'url', 'layer', '

,osm_id,osm_type,geometry,contact:website,light_rail,name,network,network:short,network:wikidata,official_name,...,description:old_name,source:old_name,internet_access:ssid,shelter,name:ko,name:ru,railway:ref:BVG,ref_name,old_name1,point
0,21302157,node,POINT (13.33645 52.51437),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Tiergarten,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Tiergarten,...,None,None,None,None,None,None,None,None,None,POINT (13.33645 52.51437)
1,26124376,node,POINT (13.28442 52.51801),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Westend,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Westend,...,None,None,None,None,None,None,None,None,None,POINT (13.28442 52.51801)
2,26603219,node,POINT (13.17981 52.42141),None,yes,Wannsee,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Wannsee S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.17981 52.42141)
3,26943869,node,POINT (13.22763 52.51015),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Pichelsberg,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin Pichelsberg,...,None,None,None,None,None,None,None,None,None,POINT (13.22763 52.51015)
4,27215654,node,POINT (13.26101 52.48825),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Grunewald,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin-Grunewald S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.26101 52.48825)


In [5]:
# Assign the osm_stations to the grid squares and create a column with the count of stations in each square and a column with the list of station names in each square
def assign_stations_to_grid(grid, stations):
    # Spatial join to assign stations to grid squares
    joined = gpd.sjoin(stations, grid, how='left', predicate='within')
    
    # Count stations in each square
    station_counts = joined.groupby('index').size().rename('station_count')
    
    # List of station names in each square
    station_names = joined.groupby('index')['name'].apply(list).rename('station_names')
    
    # Merge counts and names back to the grid
    grid = grid.merge(station_counts, left_on='index', right_index=True, how='left')
    grid = grid.merge(station_names, left_on='index', right_index=True, how='left')
    
    # Fill NaN values with 0 for counts and empty list for names
    grid['station_count'] = grid['station_count'].fillna(0).astype(int)
    grid['station_names'] = grid['station_names'].apply(lambda x: x if isinstance(x, list) else [])
    
    return grid

grid_with_stations = assign_stations_to_grid(grid, osm_stations.to_crs(grid.crs))
grid_with_stations.head()

grid_with_stations['matched_db_stations'] = np.nan  # Initialize with NaN


In [6]:
osm_stations

,osm_id,osm_type,geometry,contact:website,light_rail,name,network,network:short,network:wikidata,official_name,...,description:old_name,source:old_name,internet_access:ssid,shelter,name:ko,name:ru,railway:ref:BVG,ref_name,old_name1,point
0,21302157,node,POINT (13.33645 52.51437),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Tiergarten,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Tiergarten,...,None,None,None,None,None,None,None,None,None,POINT (13.33645 52.51437)
1,26124376,node,POINT (13.28442 52.51801),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Westend,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Westend,...,None,None,None,None,None,None,None,None,None,POINT (13.28442 52.51801)
2,26603219,node,POINT (13.17981 52.42141),None,yes,Wannsee,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Wannsee S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.17981 52.42141)
3,26943869,node,POINT (13.22763 52.51015),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Pichelsberg,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin Pichelsberg,...,None,None,None,None,None,None,None,None,None,POINT (13.22763 52.51015)
4,27215654,node,POINT (13.26101 52.48825),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Grunewald,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin-Grunewald S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.26101 52.48825)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,7590482488,node,POINT (13.54331 52.458),None,None,Hauptbahnhof,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.54331 52.458)
279,8170768521,node,POINT (13.38882 52.51699),None,None,Unter den Linden,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.38882 52.51699)
280,8170768522,node,POINT (13.40808 52.51884),None,None,Rotes Rathaus,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.40808 52.51884)
281,9082329700,node,POINT (13.39888 52.51726),None,None,Museumsinsel,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (13.39888 52.51726)


In [7]:
import re
import unicodedata

def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"\(.*?\)", "", s)          # drop parenthetical suffixes
    s = re.sub(r"^(s|u)\s+", "", s)        # drop S/U prefixes
    s = re.sub(r"^berlin[\s-]+", "", s)    # drop Berlin- / Berlin 
    s = s.replace("ß", "ss")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# First match by normalized name
db_lookup = {}
for name in db_stations['Bahnhof'].unique():
    db_lookup.setdefault(norm(name), name)

# Manual overrides for the non-exact cases where DB has a different station name
manual = {
    "Schöneweide": "Berlin-Schöneweide Pbf",
    "Berlin-Schöneweide": "Berlin-Schöneweide Pbf",
    "Wittenau": "Berlin-Wittenau (Wilhelmsruher Damm)",
    # Uncomment only if you want best-effort guesses instead of strict matching:
    # "Friedrichsfelde": "Friedrichsfelde Ost",
    # "Biesdorf-Süd": "Biesdorf",
}

osm_to_db = {
    name: manual.get(name, db_lookup.get(norm(name)))
    for name in osm_stations['name'].unique()
}

In [8]:
for osm_name, db_name in osm_to_db.items():
    if db_name is None:
        print(f"No match for OSM station '{osm_name}' with {"BVG" if osm_stations[osm_stations['name'] == osm_name]['operator'].values[0] == "Berliner Verkehrsbetriebe" else osm_stations[osm_stations['name'] == osm_name]['operator'].values[0]}")
        

No match for OSM station 'Ruhleben' with BVG
No match for OSM station 'Amrumer Straße' with BVG
No match for OSM station 'Kurfürstendamm' with BVG
No match for OSM station 'Güntzelstraße' with BVG
No match for OSM station 'Berliner Straße' with BVG
No match for OSM station 'Friedrich-Wilhelm-Platz' with BVG
No match for OSM station 'Walther-Schreiber-Platz' with BVG
No match for OSM station 'Schloßstraße' with BVG
No match for OSM station 'Theodor-Heuss-Platz' with BVG
No match for OSM station 'Neu-Westend' with BVG
No match for OSM station 'Deutsche Oper' with BVG
No match for OSM station 'Ernst-Reuter-Platz' with BVG
No match for OSM station 'Lindauer Allee' with BVG
No match for OSM station 'Bayerischer Platz' with BVG
No match for OSM station 'Wuhletal' with DB InfraGO AG
No match for OSM station 'Bismarckstraße' with BVG
No match for OSM station 'Richard-Wagner-Platz' with BVG
No match for OSM station 'Eisenacher Straße' with BVG
No match for OSM station 'Rathaus Reinickendorf' wi

In [9]:
'''
from fuzzywuzzy import fuzz

# Create a list of station names from the OSM data
osm_station_names = osm_stations['official_name'].tolist()

# Create a list of station names from the DB data
db_station_names = db_stations['Bahnhof'].tolist()

cells_with_stations = grid_with_stations[grid_with_stations['station_count'] > 0]

matches = {"idx": [], "osm_names": [], "db_name": [], "score": []}
for idx, row in cells_with_stations.iterrows():
    cell_station_names = row['station_names']
    best_match = None
    best_score = 0
    for db_name in db_station_names:
        single_best_match = None
        single_best_score = 0
        for osm_name in cell_station_names:
            score = fuzz.ratio(db_name, osm_name)
            if score > single_best_score:
                single_best_score = score
                single_best_match = db_name
        if single_best_score > best_score:
            best_score = single_best_score
            best_match = single_best_match
    matches["idx"].append(idx)
    matches["osm_names"].append(cell_station_names)
    matches["db_name"].append(best_match)
    matches["score"].append(best_score)
    
# Print the matches
matches_df = pd.DataFrame(matches)
display(matches_df)
'''

'\nfrom fuzzywuzzy import fuzz\n\n# Create a list of station names from the OSM data\nosm_station_names = osm_stations[\'official_name\'].tolist()\n\n# Create a list of station names from the DB data\ndb_station_names = db_stations[\'Bahnhof\'].tolist()\n\ncells_with_stations = grid_with_stations[grid_with_stations[\'station_count\'] > 0]\n\nmatches = {"idx": [], "osm_names": [], "db_name": [], "score": []}\nfor idx, row in cells_with_stations.iterrows():\n    cell_station_names = row[\'station_names\']\n    best_match = None\n    best_score = 0\n    for db_name in db_station_names:\n        single_best_match = None\n        single_best_score = 0\n        for osm_name in cell_station_names:\n            score = fuzz.ratio(db_name, osm_name)\n            if score > single_best_score:\n                single_best_score = score\n                single_best_match = db_name\n        if single_best_score > best_score:\n            best_score = single_best_score\n            best_match = sing

In [10]:
for idx, row in grid_with_stations.iterrows():
    if row['station_count'] > 0:
        matched_db_stations = []
        db_names_in_cell = set()
        for osm_name in row['station_names']:
            db_name = osm_to_db.get(osm_name)
            if db_name:
                matched_db_stations.append(db_name)
                db_names_in_cell.add(db_name)
        # Check for duplicates
        if len(db_names_in_cell) < len(matched_db_stations):
            print(f"Warning: Duplicate DB station matches in cell {idx} for OSM stations {row['station_names']}. Matched DB stations: {matched_db_stations}")
        
        if len(db_names_in_cell) == 1:
            grid_with_stations.at[idx, 'matched_db_stations'] = list(db_names_in_cell)[0]
        elif len(db_names_in_cell) == 0:
            print(f"No DB station match in cell {idx} for OSM stations {row['station_names']}")
        else:
            print(f"Multiple different DB station matches in cell {idx} for OSM stations {row['station_names']}.")

No DB station match in cell 479 for OSM stations ['Schloßstraße']
No DB station match in cell 519 for OSM stations ['Kaiserin-Augusta-Straße']
No DB station match in cell 753 for OSM stations ['Podbielskiallee']
No DB station match in cell 1069 for OSM stations ['Walther-Schreiber-Platz']
No DB station match in cell 1202 for OSM stations ['Breitenbachplatz']
No DB station match in cell 1295 for OSM stations ['Grenzallee']
No DB station match in cell 1402 for OSM stations ['Alt-Tempelhof']
No DB station match in cell 2235 for OSM stations ['Rüdesheimer Platz']
No DB station match in cell 2245 for OSM stations ['Friedrich-Wilhelm-Platz']
No DB station match in cell 2900 for OSM stations ['Leinestraße']
No DB station match in cell 3459 for OSM stations ['Paradestraße']
No DB station match in cell 3495 for OSM stations ['Karl-Marx-Straße']
No DB station match in cell 4017 for OSM stations ['Rathaus Schöneberg']
No DB station match in cell 4074 for OSM stations ['Boddinstraße']
No DB statio

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_51848/87596499.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Priesterweg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grid_with_stations.at[idx, 'matched_db_stations'] = list(db_names_in_cell)[0]


In [11]:
grid_with_stations['station_class'] = np.nan  # Initialize with NaN

for idx, row in grid_with_stations.iterrows():
    if row['station_count'] > 0 and not pd.isna(row['matched_db_stations']):
        db_name = row['matched_db_stations']
        db_info = db_stations[db_stations['Bahnhof'] == db_name]
        if not db_info.empty:
            station_class = db_info['klasse'].values[0]
            grid_with_stations.at[idx, 'station_class'] = int(station_class)
        else:
            print(f"DB station '{db_name}' not found in DB stations data.")
        

In [12]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    # Plot the grid colored by station class
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    grid_with_stations.plot(column='station_class', ax=ax, cmap='tab10', edgecolor='none', legend=True, alpha=1)
    grid_with_stations.plot(ax=ax, color = 'firebrick', linewidth=1, alpha=0.2)
    berlin.plot(ax=ax, edgecolor='royalblue', color='none', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    plt.title("Grid with DB Station Classes")
    plt.show()


In [13]:
# Save the grid with station info to a new Parquet file
grid_with_stations.to_parquet(PATH_PROCESSED / 'grid_with_stations.parquet')